In [ ]:
# Cell 1: Clone repo and install dependencies
import os, sys

REPO = '/content/cot_reranking'
if not os.path.exists(REPO):
    !git clone https://github.com/raheelism/cot_reranking.git {REPO}
else:
    !git -C {REPO} pull

!pip install -r {REPO}/requirements.txt -q
sys.path.insert(0, REPO)

import torch
print(f"torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
assert torch.cuda.is_available(), "Enable GPU: Runtime > Change runtime type > T4 GPU"
print("\u2713 Environment OK")

In [ ]:
# Cell 2: Mount Google Drive for results storage
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/cot_reranking_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"\u2713 Results will be saved to: {RESULTS_DIR}")

In [ ]:
# Cell 3: BEIR datasets — download, BM25, validate, save
from src.data_utils import load_beir_dataset, build_bm25_index, retrieve_bm25_top_k, save_json
from beir.retrieval.evaluation import EvaluateRetrieval

all_data = {}
for name in ['nfcorpus', 'scifact', 'trec-covid']:
    corpus, queries, qrels = load_beir_dataset(name)
    bm25, corpus_ids = build_bm25_index(corpus)
    results = retrieve_bm25_top_k(bm25, corpus_ids, queries, k=100)

    ndcg, *_ = EvaluateRetrieval.evaluate(qrels, results, [10])
    print(f"  BM25 NDCG@10: {ndcg['NDCG@10']:.4f}")
    assert ndcg['NDCG@10'] > 0.10, f"BM25 too low on {name}"

    all_data[name] = {'corpus': corpus, 'queries': queries, 'qrels': qrels}
    save_json(results,  f'{RESULTS_DIR}/{name}_bm25.json')
    save_json(qrels,    f'{RESULTS_DIR}/{name}_qrels.json')
    save_json(queries,  f'{RESULTS_DIR}/{name}_queries.json')
    print(f"\u2713 {name} done\n")

In [ ]:
# Cell 4: BRIGHT datasets — download, BM25, save
from src.data_utils import load_bright_dataset

for subset in ['biology', 'economics']:
    name = f'bright_{subset}'
    corpus, queries, qrels = load_bright_dataset(subset)
    bm25, corpus_ids = build_bm25_index(corpus)
    results = retrieve_bm25_top_k(bm25, corpus_ids, queries, k=100)

    all_data[name] = {'corpus': corpus, 'queries': queries, 'qrels': qrels}
    save_json(results,  f'{RESULTS_DIR}/{name}_bm25.json')
    save_json(qrels,    f'{RESULTS_DIR}/{name}_qrels.json')
    save_json(queries,  f'{RESULTS_DIR}/{name}_queries.json')
    print(f"\u2713 {name} done\n")

print("\u2713 All BM25 results saved to Drive")